# Extracción de videos de YouTube (YouTube Data API v3)

Genera `youtube_videos_largos_<fecha>.csv` con las mismas 12 columnas y el mismo formato que `youtube_videos_largos_20260424_143458.csv`, más `extraido_el` (fecha de la extracción, que el notebook de análisis usa para calcular la antigüedad).

**Pasos**
1. 🔑 *Secretos* (barra izquierda) → agrega `YT_API_KEY` con tu clave y activa *Acceso del notebook*. Si no hay secreto, la celda 2 te pide la clave.
2. Celda 1: revisa la lista `CANALES` (meta: 25 canales únicos).
3. Corre las celdas en orden. La celda 4 solo valida los handles; la 5 extrae.

**Cuota:** no se usa `search.list`. Cada canal cuesta 1 unidad más 1 por cada página de 50 videos en `playlistItems.list` y `videos.list` (≈ 300–600 unidades de 10,000 diarias para 25 canales).

In [ ]:
# CELDA 1 — Parámetros y lista de canales (@handle o identificador que empieza con UC)
CANALES = [
    # 15 canales en español del corpus anterior
    "@BigDataNews", "@AtomixNoticias", "@BehindtheGames", "@eurogamerspain", "@SrBaityBait",
    "@Dan-PuertaAlSotano", "@SemonsChannel", "@soy_alexwhite", "@GNGamesOficial", "@ELGEORGENEW",
    "@extas1s", "@levelupcom", "@CVJExtra", "@ElLoconGamer", "@RayBacon",
    # Canales nuevos
    "@GamerDeNacimiento", "@3DJuegosmx", "@michaelquesada", "@LaCavernaDelGamer",
    "@gamerdentista", "@NintendoFiles",
    "@SEGUNELGEORGE",  # mismo creador que @ELGEORGENEW: conserva solo uno de los dos
    "@CulturaVJ",      # misma marca que @CVJExtra: conserva solo uno de los dos
    # Faltan canales para llegar a 25:
    # "@...", "@...",
]

VIDEOS_POR_CANAL   = 150   # videos largos a conservar por canal (los más recientes que cumplan los filtros)
DURACION_MIN_SEG   = 180   # desde octubre de 2024 los Shorts pueden durar hasta 3 minutos
DIAS_MIN_PUBLICADO = 7     # descarta videos con menos de 7 días: sus vistas todavía no se estabilizan
MAX_PAGINAS        = 40    # tope de páginas de 50 subidas por canal (2,000 videos)

In [ ]:
# CELDA 2 — Conexión a la API
!pip install -q google-api-python-client
import re, time
import pandas as pd
from datetime import datetime, timezone
from getpass import getpass
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from google.colab import userdata, files

try:
    API_KEY = userdata.get("YT_API_KEY")
except Exception:
    API_KEY = getpass("Pega tu clave de la YouTube Data API: ")
# Algunas versiones de Colab devuelven la respuesta completa del panel de secretos en lugar de solo el valor
if isinstance(API_KEY, dict):
    API_KEY = API_KEY.get("data", API_KEY).get("payload")
API_KEY = str(API_KEY).strip()
assert API_KEY.startswith("AIza"), "La clave no tiene el formato esperado (debe empezar con AIza)."

yt = build("youtube", "v3", developerKey=API_KEY, cache_discovery=False)
AHORA = datetime.now(timezone.utc)
print("Conectado. Fecha de extracción (UTC):", AHORA.strftime("%Y-%m-%d %H:%M:%S"))

In [ ]:
# CELDA 3 — Funciones
COLUMNAS = ["input_identifier", "channel_id", "channel_title", "video_id", "title", "published_at",
            "duration_seconds", "views", "likes", "comments", "tags", "description", "extraido_el"]

def duracion_seg(iso):
    # Convierte la duración ISO 8601 de la API (p. ej. PT1H2M3S) a segundos
    m = re.fullmatch(r"P(?:(\d+)D)?T?(?:(\d+)H)?(?:(\d+)M)?(?:(\d+)S)?", iso or "")
    if not m:
        return None
    d, h, mi, s = (int(x) if x else 0 for x in m.groups())
    return d * 86400 + h * 3600 + mi * 60 + s

def llamar(req, intentos=4):
    for i in range(intentos):
        try:
            return req.execute()
        except HttpError as e:
            if e.resp.status in (403, 429) and "quota" in str(e).lower():
                raise RuntimeError("Se agotó la cuota diaria de la API; vuelve a correr mañana.") from e
            if i == intentos - 1:
                raise
            time.sleep(2 ** i)

def info_canal(ident):
    ident = ident.strip()
    parte = "snippet,contentDetails,statistics"
    if re.fullmatch(r"UC[\w-]{22}", ident):
        r = llamar(yt.channels().list(part=parte, id=ident))
    else:
        r = llamar(yt.channels().list(part=parte, forHandle=ident if ident.startswith("@") else "@" + ident))
    if not r.get("items"):
        return None
    c = r["items"][0]
    return dict(channel_id=c["id"], channel_title=c["snippet"]["title"],
                uploads=c["contentDetails"]["relatedPlaylists"]["uploads"],
                suscriptores=c["statistics"].get("subscriberCount"), pais=c["snippet"].get("country"))

def fecha(iso):
    # Mismo formato que el CSV anterior: "2026-04-24 00:34:05"
    return pd.Timestamp(iso).tz_convert("UTC").strftime("%Y-%m-%d %H:%M:%S")

def videos_canal(canal, ident):
    filas, token, paginas = [], None, 0
    descartes = dict(cortos=0, recientes=0, en_vivo=0, sin_vistas=0)
    while len(filas) < VIDEOS_POR_CANAL and paginas < MAX_PAGINAS:
        r = llamar(yt.playlistItems().list(part="contentDetails", playlistId=canal["uploads"],
                                           maxResults=50, pageToken=token))
        ids = [it["contentDetails"]["videoId"] for it in r.get("items", [])]
        paginas += 1
        if ids:
            v = llamar(yt.videos().list(part="snippet,contentDetails,statistics,liveStreamingDetails", id=",".join(ids)))
            for it in v.get("items", []):
                sn, st = it["snippet"], it.get("statistics", {})
                dur = duracion_seg(it["contentDetails"].get("duration"))
                if sn.get("liveBroadcastContent", "none") != "none" or "liveStreamingDetails" in it:
                    descartes["en_vivo"] += 1; continue
                if dur is None or dur < DURACION_MIN_SEG:
                    descartes["cortos"] += 1; continue
                if (AHORA - pd.Timestamp(sn["publishedAt"])).days < DIAS_MIN_PUBLICADO:
                    descartes["recientes"] += 1; continue
                if "viewCount" not in st:
                    descartes["sin_vistas"] += 1; continue
                filas.append(dict(
                    input_identifier=ident, channel_id=canal["channel_id"], channel_title=canal["channel_title"],
                    video_id=it["id"], title=sn["title"], published_at=fecha(sn["publishedAt"]),
                    duration_seconds=dur, views=int(st["viewCount"]),
                    likes=int(st["likeCount"]) if "likeCount" in st else None,
                    comments=int(st["commentCount"]) if "commentCount" in st else None,
                    tags=", ".join(sn.get("tags", [])), description=sn.get("description", ""),
                    extraido_el=AHORA.strftime("%Y-%m-%d %H:%M:%S")))
                if len(filas) >= VIDEOS_POR_CANAL:
                    break
        token = r.get("nextPageToken")
        if not token:
            break
    return filas, descartes

In [ ]:
# CELDA 4 — Verificar los handles antes de extraer (≈ 2 unidades por canal)
ver = []
for h in dict.fromkeys(CANALES):
    c = info_canal(h)
    if c is None:
        ver.append(dict(handle=h, canal="NO ENCONTRADO")); continue
    r = llamar(yt.playlistItems().list(part="contentDetails", playlistId=c["uploads"], maxResults=50))
    fechas = [it["contentDetails"].get("videoPublishedAt") for it in r.get("items", [])]
    fechas = [f for f in fechas if f]
    ver.append(dict(handle=h, canal=c["channel_title"], channel_id=c["channel_id"], pais=c["pais"],
                    suscriptores=int(c["suscriptores"] or 0), ultimo_video=max(fechas)[:10] if fechas else None))
ver = pd.DataFrame(ver)
print(ver.to_string(index=False))
print(f"\nCanales únicos encontrados: {ver['channel_id'].nunique()}")
rep = ver[ver["channel_id"].notna() & ver.duplicated("channel_id", keep=False)]
if len(rep):
    print("Handles que apuntan al mismo canal:", rep["handle"].tolist())

In [ ]:
# CELDA 5 — Extracción
todas, resumen, vistos = [], [], set()
for ident in dict.fromkeys(CANALES):           # quita handles repetidos conservando el orden
    canal = info_canal(ident)
    if canal is None:
        print(f"✗ {ident}: no se encontró el canal (revisa el handle)"); continue
    if canal["channel_id"] in vistos:
        print(f"✗ {ident}: es el mismo canal que uno ya extraído ({canal['channel_title']})"); continue
    vistos.add(canal["channel_id"])
    filas, desc = videos_canal(canal, ident)
    todas += filas
    resumen.append(dict(input_identifier=ident, channel_id=canal["channel_id"], channel_title=canal["channel_title"],
                        pais=canal["pais"], suscriptores=canal["suscriptores"], videos=len(filas), **desc))
    print(f"✓ {canal['channel_title']:<30} {len(filas):>4} videos | descartados: {desc}")

df = pd.DataFrame(todas, columns=COLUMNAS).drop_duplicates("video_id").reset_index(drop=True)
res = pd.DataFrame(resumen)
print(f"\nTotal: {len(df):,} videos únicos de {df['channel_id'].nunique()} canales")
inc = res[res["videos"] < VIDEOS_POR_CANAL]
print("Canales que no llegaron al tope:\n" + inc[["channel_title", "videos"]].to_string(index=False) if len(inc) else "Todos los canales llegaron al tope.")

In [ ]:
# CELDA 6 — Idioma de cada canal (misma regla que el notebook de análisis)
STOP_ES = set("de la el en y que los las un una por con para del es se no su al lo más como ya ser este esta nuevo nueva todo sin muy qué así ha".split())
STOP_EN = set("the of and to in is for on with you this that it are be at your new how why what was not all from best can will".split())
cuenta = lambda t, S: sum(w in S for w in re.findall(r"[a-záéíóúñü]+", str(t).lower()))
idi = df.assign(es=df["title"].map(lambda t: cuenta(t, STOP_ES)), en=df["title"].map(lambda t: cuenta(t, STOP_EN)))
idi = idi.groupby("channel_title")[["es", "en"]].sum()
idi["prop_es"] = (idi["es"] / (idi["es"] + idi["en"])).round(2)
print(idi.sort_values("prop_es").to_string())
print("\nCanales con prop_es <= 0.5 (quedarían fuera del análisis):", idi.index[idi["prop_es"] <= 0.5].tolist())

In [ ]:
# CELDA 7 — Guardar y descargar
sello = AHORA.strftime("%Y%m%d_%H%M%S")
f_videos, f_res = f"youtube_videos_largos_{sello}.csv", f"resumen_extraccion_{sello}.csv"
df.to_csv(f_videos, index=False, encoding="utf-8")
res.to_csv(f_res, index=False, encoding="utf-8")
print(f_videos, df.shape)
files.download(f_videos); files.download(f_res)